*Migrated to NotebookSession API in Phase 4c-i — see notebooks/UTIL_README.md*

# ADP1 BERDL Analysis

This notebook loads experimental data into the BERDL (Biological Entity Relationship Data Lake) database and correlates proteomics, essentiality, and mutant growth rate data with fitness scores and essentiality predictions from the gene_phenotypes table.

## Data Sources
1. **BERDL Database** (`berdl_tables.db`) - Contains gene phenotypes, genome features, etc.
2. **Proteomics Data** (`ASCR_UGA_Proteomics_DgoA_add_strains_2025_pyruvate.xlsx`) - Protein expression data
3. **Strain Information** (`ADP1_Synbio_LIMS.xlsx`) - Strain metadata from LIMS
4. **Essentiality Data** (`essentiality_gene_lists.json`) - Gene essentiality classifications
5. **Mutant Growth Rates** (`MutantGrowthRatesData.xls`) - Growth rate measurements for mutants

## Objectives
- Load proteomics data (individual replicates and strain averages) into the database
- Create a strains table from LIMS data
- Load essentiality data into gene_features table
- Load mutant growth rate data into gene_features table
- Correlate experimental data with phenotype predictions

## Initialize Utilities and Database Connection

Load the util module and establish connection to the BERDL SQLite database.

In [1]:
%run util.py
import sqlite3
from scipy import stats

# Database path
db_path = "data/berdl_tables.db"

# Verify database exists and show tables
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = [row[0] for row in cursor.fetchall()]
print(f"Database tables: {tables}")
conn.close()

session.cache.save('ADP1BERDLAnalysis', {'db_path': db_path, 'tables': tables})

Database tables: ['genome', 'genome_ani', 'genome_features', 'missing_functions', 'pan_genome_features', 'gene_reaction_data', 'genome_reactions', 'growth_phenotype_summary', 'growth_phenotypes_detailed', 'gene_phenotypes', 'ontology_terms', 'ontology_definitions', 'ontology_relationships', 'gene_essentiality', 'strains']


[KBUtilLib] Failed to import rcsb_pdb_utils: ModuleNotFoundError: No module named 'aiohttp'


CacheEntry(id='ADP1BERDLAnalysis', type='dict', blob_path='blobs/86106704512e3f700b4a9661491bda790c58496f8c5a0a98940d96ef7201f8b3.json', content_hash='86106704512e3f700b4a9661491bda790c58496f8c5a0a98940d96ef7201f8b3', n_bytes=423, metadata=None, created_at=datetime.datetime(2026, 5, 6, 21, 55, 25, 644867, tzinfo=datetime.timezone.utc))

## Load and Process Proteomics Data

Load proteomics data from the Excel file. We load:
1. All individual replicate columns
2. Computed averages for each strain

The data includes protein expression measurements for strains: ADP1, ACN2586, ACN2821, ACN3425, ACN3427, ACN3429, ACN3430

In [2]:
%run util.py


# Load proteomics data
proteomics_file = "data/ASCR_UGA_Proteomics_DgoA_add_strains_2025_pyruvate.xlsx"
xl = pd.ExcelFile(proteomics_file)

# Load metadata to understand strain structure
metadata_df = xl.parse('Metadata')
strains = metadata_df['Strain'].unique().tolist()
print(f"Strains in proteomics data: {strains}")

# Load imputed data (main expression values)
imputed_df = xl.parse('Imputed')
print(f"\nImputed data shape: {imputed_df.shape}")
print(f"Columns: {list(imputed_df.columns)}")

# Clean up gene IDs - remove '>' prefix from FASTA headers
imputed_df['gene_id'] = imputed_df['FASTA.Title.Lines'].str.replace('>', '', regex=False)

# Get all replicate columns (exclude metadata columns)
replicate_cols = [col for col in imputed_df.columns if col.startswith('Pyruvate_')]
print(f"\nReplicate columns ({len(replicate_cols)}): {replicate_cols}")

# Compute strain averages
strain_averages = {}
for strain in strains:
    strain_cols = [col for col in replicate_cols if f'_{strain}_' in col]
    if strain_cols:
        strain_averages[strain] = imputed_df[strain_cols].mean(axis=1)
        print(f"  {strain}: {len(strain_cols)} replicates, avg expression range: {strain_averages[strain].min():.2f} - {strain_averages[strain].max():.2f}")

# Create proteomics DataFrame with gene_id, all replicates, and averages
proteomics_data = imputed_df[['gene_id', 'ACIAD'] + replicate_cols].copy()

# Add average columns
for strain, avg_values in strain_averages.items():
    proteomics_data[f'avg_{strain}'] = avg_values

print(f"\nFinal proteomics data shape: {proteomics_data.shape}")
print(proteomics_data.head())

# Save for later use
session.cache.save('proteomics_data', {
    'strains': strains,
    'replicate_columns': replicate_cols,
    'gene_count': len(proteomics_data),
    'sample_genes': proteomics_data['gene_id'].head(10).tolist()
})

Strains in proteomics data: ['ADP1', 'ACN2821', 'ACN3430', 'ACN3425', 'ACN2586', 'ACN3429', 'ACN3427']

Imputed data shape: (2383, 37)
Columns: ['FASTA.Title.Lines', 'ACIAD', 'Pyruvate_ACN2586_DgoA_2025_A', 'Pyruvate_ACN2586_DgoA_2025_B', 'Pyruvate_ACN2586_DgoA_2025_C', 'Pyruvate_ACN2586_DgoA_2025_D', 'Pyruvate_ACN2586_DgoA_2025_E', 'Pyruvate_ACN2821_DgoA_2025_E', 'Pyruvate_ACN2821_DgoA_2025_A', 'Pyruvate_ACN2821_DgoA_2025_B', 'Pyruvate_ACN2821_DgoA_2025_C', 'Pyruvate_ACN2821_DgoA_2025_D', 'Pyruvate_ACN3425_DgoA_2025_A', 'Pyruvate_ACN3425_DgoA_2025_B', 'Pyruvate_ACN3425_DgoA_2025_C', 'Pyruvate_ACN3425_DgoA_2025_D', 'Pyruvate_ACN3425_DgoA_2025_E', 'Pyruvate_ACN3427_DgoA_2025_A', 'Pyruvate_ACN3427_DgoA_2025_B', 'Pyruvate_ACN3427_DgoA_2025_C', 'Pyruvate_ACN3427_DgoA_2025_D', 'Pyruvate_ACN3427_DgoA_2025_E', 'Pyruvate_ACN3429_DgoA_2025_A', 'Pyruvate_ACN3429_DgoA_2025_B', 'Pyruvate_ACN3429_DgoA_2025_C', 'Pyruvate_ACN3429_DgoA_2025_D', 'Pyruvate_ACN3429_DgoA_2025_E', 'Pyruvate_ACN3430_DgoA_20

/home/chenry/VirtualEnvironments/kbu.nb-adp1notebooks-py3.10/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


CacheEntry(id='proteomics_data', type='dict', blob_path='blobs/1e7b3064122c4cbabd34bb7ad8206186f30cf44436805de7d4413ed82d812dde.json', content_hash='1e7b3064122c4cbabd34bb7ad8206186f30cf44436805de7d4413ed82d812dde', n_bytes=3061, metadata=None, created_at=datetime.datetime(2026, 5, 6, 21, 55, 39, 155342, tzinfo=datetime.timezone.utc))

## Add Proteomics Data to genome_features Table

Add proteomics data directly to the genome_features table by mapping the ACIAD column from proteomics to the feature_id in genome_features. This includes:
- Individual replicate values for each strain
- Computed average values for each strain

For genes in proteomics that don't exist in genome_features, we add them as new rows.

In [3]:
%run util.py
import sqlite3


# Load saved data
saved_data = session.cache.load('proteomics_data')
strains = saved_data['strains']
replicate_cols = saved_data['replicate_columns']

# Reload proteomics data
proteomics_file = "data/ASCR_UGA_Proteomics_DgoA_add_strains_2025_pyruvate.xlsx"
xl = pd.ExcelFile(proteomics_file)
imputed_df = xl.parse('Imputed')
imputed_df['gene_id'] = imputed_df['FASTA.Title.Lines'].str.replace('>', '', regex=False)

# Create final DataFrame with averages
proteomics_data = imputed_df[['gene_id', 'ACIAD'] + replicate_cols].copy()
for strain in strains:
    strain_cols = [col for col in replicate_cols if f'_{strain}_' in col]
    if strain_cols:
        proteomics_data[f'avg_{strain}'] = imputed_df[strain_cols].mean(axis=1)

# Connect to database
db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Delete proteomics_pyruvate table if it exists (no longer creating reference table)
conn.execute("DROP TABLE IF EXISTS proteomics_pyruvate")
print("Removed proteomics_pyruvate reference table (if existed)")

# Get existing genome_features data and build mapping from ACIAD numeric to feature_id
cursor.execute("SELECT feature_id FROM genome_features WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'")
existing_features = set(row[0] for row in cursor.fetchall())
print(f"Existing features in genome_features: {len(existing_features)}")

# Build mapping: ACIAD numeric part -> feature_id (e.g., "16655" -> "ACIAD_RS16655")
aciad_to_feature = {}
for feature_id in existing_features:
    if feature_id and feature_id.startswith('ACIAD_RS'):
        num_part = feature_id.replace('ACIAD_RS', '')
        aciad_to_feature[num_part] = feature_id
        try:
            aciad_to_feature[str(int(num_part))] = feature_id
        except:
            pass

print(f"ACIAD to feature_id mappings: {len(aciad_to_feature)}")

# Add proteomics columns to genome_features if they don't exist
cursor.execute("PRAGMA table_info(genome_features)")
existing_cols = [col[1] for col in cursor.fetchall()]

# Add average columns for each strain
avg_cols_to_add = [f'proteomics_avg_{strain}' for strain in strains]
for col in avg_cols_to_add:
    if col not in existing_cols:
        conn.execute(f"ALTER TABLE genome_features ADD COLUMN {col} REAL")
        print(f"Added {col} column to genome_features")

# Also add a column for proteomics gene_id (the FASTA header name like DgoA, etc.)
if 'proteomics_gene_id' not in existing_cols:
    conn.execute("ALTER TABLE genome_features ADD COLUMN proteomics_gene_id VARCHAR(255)")
    print("Added proteomics_gene_id column to genome_features")

# Map proteomics data to feature_ids and update/insert into genome_features
update_count = 0
insert_count = 0
unmapped_count = 0

for _, row in proteomics_data.iterrows():
    aciad_val = row['ACIAD']
    proteomics_gene_id = row['gene_id']
    
    # Try to map ACIAD to feature_id
    feature_id = None
    if pd.notna(aciad_val):
        aciad_str = str(int(aciad_val)) if isinstance(aciad_val, float) else str(aciad_val)
        feature_id = aciad_to_feature.get(aciad_str)
    
    # Build update values
    avg_values = {f'proteomics_avg_{strain}': row.get(f'avg_{strain}') for strain in strains}
    
    if feature_id and feature_id in existing_features:
        # Update existing row
        set_clause = ', '.join([f"{k} = ?" for k in avg_values.keys()])
        values = list(avg_values.values()) + [proteomics_gene_id, feature_id]
        cursor.execute(f"""
            UPDATE genome_features 
            SET {set_clause}, proteomics_gene_id = ?
            WHERE feature_id = ? AND genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
        """, values)
        if cursor.rowcount > 0:
            update_count += 1
    else:
        # Insert new row for unmapped proteomics gene
        # Use proteomics_gene_id as feature_id if no ACIAD mapping
        new_feature_id = feature_id if feature_id else f"proteomics_{proteomics_gene_id}"
        
        cols = ['genome_id', 'contig_id', 'feature_id', 'length', 'proteomics_gene_id'] + list(avg_values.keys())
        vals = ['user_Acinetobacter_baylyi_ADP1_RAST', 'proteomics_unmapped', new_feature_id, 0, proteomics_gene_id] + list(avg_values.values())
        
        placeholders = ', '.join(['?' for _ in vals])
        col_names = ', '.join(cols)
        
        try:
            cursor.execute(f"INSERT INTO genome_features ({col_names}) VALUES ({placeholders})", vals)
            insert_count += 1
        except sqlite3.IntegrityError:
            # If there's a unique constraint violation, try to update instead
            unmapped_count += 1

conn.commit()

print(f"\nUpdated {update_count} existing rows in genome_features with proteomics data")
print(f"Inserted {insert_count} new rows for proteomics genes not in genome_features")
if unmapped_count > 0:
    print(f"Skipped {unmapped_count} rows due to constraint violations")

# Show sample of updated data
sample_df = pd.read_sql("""
    SELECT feature_id, proteomics_gene_id, proteomics_avg_ADP1, proteomics_avg_ACN2586 
    FROM genome_features 
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
    AND proteomics_avg_ADP1 IS NOT NULL 
    LIMIT 10
""", conn)
print("\nSample genome_features with proteomics data:")
print(sample_df)

# Show newly inserted proteomics genes
if insert_count > 0:
    new_df = pd.read_sql("""
        SELECT feature_id, proteomics_gene_id, proteomics_avg_ADP1
        FROM genome_features 
        WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
        AND contig_id = 'proteomics_unmapped'
        LIMIT 5
    """, conn)
    print("\nNewly inserted proteomics genes:")
    print(new_df)

conn.close()

session.cache.save('proteomics_added_to_genome_features', {
    'total_proteomics_genes': len(proteomics_data),
    'updated_rows': update_count,
    'inserted_rows': insert_count,
    'unmapped_count': unmapped_count,
    'columns_added': avg_cols_to_add + ['proteomics_gene_id']
})

Removed proteomics_pyruvate reference table (if existed)
Existing features in genome_features: 5852
ACIAD to feature_id mappings: 5157

Updated 0 existing rows in genome_features with proteomics data
Inserted 0 new rows for proteomics genes not in genome_features
Skipped 2383 rows due to constraint violations

Sample genome_features with proteomics data:
                                          feature_id  \
0                                    proteomics_DgoA   
1                                 proteomics_DgoA_Ec   
2                                     proteomics_Kan   
3  proteomics_lcl|NC_005966.1_prot_ACIAD_RS07445_...   
4  proteomics_lcl|NC_005966.1_prot_ACIAD_RS17085_...   
5  proteomics_lcl|NC_005966.1_prot_WP_000048256.1...   
6  proteomics_lcl|NC_005966.1_prot_WP_000076440.1...   
7  proteomics_lcl|NC_005966.1_prot_WP_000124858.1...   
8  proteomics_lcl|NC_005966.1_prot_WP_000424060.1...   
9  proteomics_lcl|NC_005966.1_prot_WP_000831329.1...   

                          

/home/chenry/VirtualEnvironments/kbu.nb-adp1notebooks-py3.10/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


CacheEntry(id='proteomics_added_to_genome_features', type='dict', blob_path='blobs/b245e6c180d7e20840741264ea459dc7731b24108ad3fe40b4279cfb0bb77999.json', content_hash='b245e6c180d7e20840741264ea459dc7731b24108ad3fe40b4279cfb0bb77999', n_bytes=363, metadata=None, created_at=datetime.datetime(2026, 5, 6, 21, 55, 51, 63281, tzinfo=datetime.timezone.utc))

## Load Strains Data from LIMS

Load strain information from the ADP1_Synbio_LIMS.xlsx file and filter to strains used in the proteomics experiment.

In [4]:
%run util.py


# Load LIMS data
lims_file = "data/ADP1_Synbio_LIMS.xlsx"
xl = pd.ExcelFile(lims_file)
strains_df = xl.parse('Strains')

print(f"Total strains in LIMS: {len(strains_df)}")
print(f"Columns: {list(strains_df.columns)}")

# Load proteomics strains
proteomics_info = session.cache.load('proteomics_data')
proteomics_strains = proteomics_info['strains']
print(f"\nProteomics strains: {proteomics_strains}")

# Filter LIMS strains to those in proteomics data
proteomics_strains_df = strains_df[strains_df['Name'].isin(proteomics_strains)].copy()
print(f"\nStrains found in LIMS matching proteomics: {len(proteomics_strains_df)}")
print(proteomics_strains_df[['Name', 'Genotype', 'Phenotype', 'Description', 'dgoA allele']].to_string())

session.cache.save('strains_lims_data', {
    'total_lims_strains': len(strains_df),
    'proteomics_strains_found': len(proteomics_strains_df),
    'strain_names': proteomics_strains_df['Name'].tolist()
})

Total strains in LIMS: 1020
Columns: ['Database ID', 'Name', 'Strain construction', 'Genotype', 'Phenotype', 'Description', 'Parent strain', 'Supporting documents', 'Reference genome (GenBank file)', 'Reference genome (SnapGene file)', 'Reference genome (protein fasta)', 'Reference genome created by', 'Reference genome verified?', 'Multiple genotypes (population?)', 'dgoA allele', 'dgoA-Kan copy number', 'Synthetic Bridging Fragment', 'Error']

Proteomics strains: ['ADP1', 'ACN2821', 'ACN3430', 'ACN3425', 'ACN2586', 'ACN3429', 'ACN3427']

Strains found in LIMS matching proteomics: 7
       Name                                                                                                                                         Genotype                                           Phenotype                                                                                                                                                                                                          

CacheEntry(id='strains_lims_data', type='dict', blob_path='blobs/c534600a4999db2b76ac35e44b98b9e0a01cdc09fa23ea3bc63afca9f103c367.json', content_hash='c534600a4999db2b76ac35e44b98b9e0a01cdc09fa23ea3bc63afca9f103c367', n_bytes=191, metadata=None, created_at=datetime.datetime(2026, 5, 6, 21, 55, 56, 95718, tzinfo=datetime.timezone.utc))

## Create Strains Table in Database

Store strain information in a new table in the BERDL database. Include all strains from LIMS that appear in the proteomics data.

In [5]:
%run util.py
import sqlite3


# Reload LIMS data
lims_file = "data/ADP1_Synbio_LIMS.xlsx"
xl = pd.ExcelFile(lims_file)
strains_df = xl.parse('Strains')

# Get proteomics strains
proteomics_info = session.cache.load('proteomics_data')
proteomics_strains = proteomics_info['strains']

# Filter and clean strains data
proteomics_strains_df = strains_df[strains_df['Name'].isin(proteomics_strains)].copy()

# Select relevant columns and rename for database
strains_table = proteomics_strains_df[[
    'Database ID', 'Name', 'Strain construction', 'Genotype', 'Phenotype', 
    'Description', 'Parent strain', 'dgoA allele', 'dgoA-Kan copy number'
]].copy()

# Rename columns to be database-friendly
strains_table.columns = [
    'database_id', 'name', 'strain_construction', 'genotype', 'phenotype',
    'description', 'parent_strain', 'dgoa_allele', 'dgoa_kan_copy_number'
]

# Connect to database
db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)

# Drop table if exists and create new
conn.execute("DROP TABLE IF EXISTS strains")
strains_table.to_sql('strains', conn, index=False)

# Create index on name
conn.execute("CREATE INDEX IF NOT EXISTS idx_strains_name ON strains(name)")

# Verify
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM strains")
count = cursor.fetchone()[0]
print(f"Strains table created with {count} rows")

# Show data
sample_df = pd.read_sql("SELECT * FROM strains", conn)
print("\nStrains data:")
print(sample_df.to_string())

conn.close()

session.cache.save('strains_table_created', {'row_count': count, 'columns': list(strains_table.columns)})

Strains table created with 7 rows

Strains data:
  database_id     name                                                                                                                                                                                               strain_construction                                                                                                                                         genotype                                           phenotype                                                                                                                                                                                                                    description parent_strain dgoa_allele dgoa_kan_copy_number
0        None     ADP1                                                                                Neidle lab wildtype strain of Acinetobacter baylyi. Mutations relative to RefSeq NC_005966 can be found in 'Supporting documents'.                   

CacheEntry(id='strains_table_created', type='dict', blob_path='blobs/feb39bed945f3571c4bc19caed2854094c090b046b71305020908415d6b47891.json', content_hash='feb39bed945f3571c4bc19caed2854094c090b046b71305020908415d6b47891', n_bytes=217, metadata=None, created_at=datetime.datetime(2026, 5, 6, 21, 56, 7, 512496, tzinfo=datetime.timezone.utc))

## Load Essentiality Data

Load gene essentiality classifications from the cached essentiality_gene_lists.json file. This contains:
- essential_minimal: Genes essential in minimal media
- essential_lb: Genes essential in LB media
- uncertain_minimal/lb: Genes with uncertain essentiality
- dispensable_minimal/lb: Genes that are dispensable

In [6]:
%run util.py



# Load essentiality data
essentiality_data = session.cache.load('essentiality_gene_lists')

# Summarize
print("Essentiality categories:")
for key, genes in essentiality_data.items():
    print(f"  {key}: {len(genes)} genes")

# Create a mapping of gene_id to essentiality status
gene_essentiality = {}

# Assign essentiality status (prioritize essential > uncertain > dispensable)
for gene in essentiality_data.get('essential_minimal', []):
    gene_essentiality[gene] = {'minimal_media': 'essential'}
for gene in essentiality_data.get('essential_lb', []):
    if gene not in gene_essentiality:
        gene_essentiality[gene] = {}
    gene_essentiality[gene]['lb_media'] = 'essential'

for gene in essentiality_data.get('uncertain_minimal', []):
    if gene not in gene_essentiality:
        gene_essentiality[gene] = {}
    if 'minimal_media' not in gene_essentiality[gene]:
        gene_essentiality[gene]['minimal_media'] = 'uncertain'
for gene in essentiality_data.get('uncertain_lb', []):
    if gene not in gene_essentiality:
        gene_essentiality[gene] = {}
    if 'lb_media' not in gene_essentiality[gene]:
        gene_essentiality[gene]['lb_media'] = 'uncertain'

for gene in essentiality_data.get('dispensable_minimal', []):
    if gene not in gene_essentiality:
        gene_essentiality[gene] = {}
    if 'minimal_media' not in gene_essentiality[gene]:
        gene_essentiality[gene]['minimal_media'] = 'dispensable'
for gene in essentiality_data.get('dispensable_lb', []):
    if gene not in gene_essentiality:
        gene_essentiality[gene] = {}
    if 'lb_media' not in gene_essentiality[gene]:
        gene_essentiality[gene]['lb_media'] = 'dispensable'

print(f"\nTotal genes with essentiality data: {len(gene_essentiality)}")

# Create DataFrame for database
essentiality_rows = []
for gene_id, status in gene_essentiality.items():
    essentiality_rows.append({
        'gene_id': gene_id,
        'essentiality_minimal': status.get('minimal_media'),
        'essentiality_lb': status.get('lb_media')
    })

essentiality_df = pd.DataFrame(essentiality_rows)
print("\nEssentiality DataFrame:")
print(essentiality_df.head(10))

session.cache.save('essentiality_processed', {
    'total_genes': len(gene_essentiality),
    'categories': list(essentiality_data.keys())
})

KeyError: "Cache object 'essentiality_gene_lists' not found"

## Add Essentiality Data to genome_features Table

Add essentiality data directly to the genome_features table. For genes that don't exist in the table, we'll add them as new rows with empty values for other columns.

In [ ]:
%run util.py
import sqlite3



# Reload essentiality data
essentiality_data = session.cache.load('essentiality_gene_lists')

# Rebuild gene_essentiality mapping
gene_essentiality = {}
for gene in essentiality_data.get('essential_minimal', []):
    gene_essentiality[gene] = {'minimal_media': 'essential'}
for gene in essentiality_data.get('essential_lb', []):
    if gene not in gene_essentiality:
        gene_essentiality[gene] = {}
    gene_essentiality[gene]['lb_media'] = 'essential'
for gene in essentiality_data.get('uncertain_minimal', []):
    if gene not in gene_essentiality:
        gene_essentiality[gene] = {}
    if 'minimal_media' not in gene_essentiality[gene]:
        gene_essentiality[gene]['minimal_media'] = 'uncertain'
for gene in essentiality_data.get('uncertain_lb', []):
    if gene not in gene_essentiality:
        gene_essentiality[gene] = {}
    if 'lb_media' not in gene_essentiality[gene]:
        gene_essentiality[gene]['lb_media'] = 'uncertain'
for gene in essentiality_data.get('dispensable_minimal', []):
    if gene not in gene_essentiality:
        gene_essentiality[gene] = {}
    if 'minimal_media' not in gene_essentiality[gene]:
        gene_essentiality[gene]['minimal_media'] = 'dispensable'
for gene in essentiality_data.get('dispensable_lb', []):
    if gene not in gene_essentiality:
        gene_essentiality[gene] = {}
    if 'lb_media' not in gene_essentiality[gene]:
        gene_essentiality[gene]['lb_media'] = 'dispensable'

print(f"Total genes with essentiality data: {len(gene_essentiality)}")

# Connect to database
db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Add columns to genome_features if they don't exist
cursor.execute("PRAGMA table_info(genome_features)")
existing_cols = [col[1] for col in cursor.fetchall()]

if 'essentiality_minimal' not in existing_cols:
    conn.execute("ALTER TABLE genome_features ADD COLUMN essentiality_minimal VARCHAR(20)")
    print("Added essentiality_minimal column to genome_features")
if 'essentiality_lb' not in existing_cols:
    conn.execute("ALTER TABLE genome_features ADD COLUMN essentiality_lb VARCHAR(20)")
    print("Added essentiality_lb column to genome_features")

# Get existing feature_ids in genome_features
cursor.execute("SELECT feature_id FROM genome_features WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'")
existing_features = set(row[0] for row in cursor.fetchall())
print(f"Existing features in genome_features: {len(existing_features)}")

# Update existing rows and track genes that need to be inserted
update_count = 0
genes_to_insert = []

for gene_id, status in gene_essentiality.items():
    essentiality_minimal = status.get('minimal_media')
    essentiality_lb = status.get('lb_media')
    
    if gene_id in existing_features:
        # Update existing row
        cursor.execute("""
            UPDATE genome_features 
            SET essentiality_minimal = ?, essentiality_lb = ?
            WHERE feature_id = ? AND genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
        """, (essentiality_minimal, essentiality_lb, gene_id))
        if cursor.rowcount > 0:
            update_count += 1
    else:
        # Track for insertion
        genes_to_insert.append({
            'gene_id': gene_id,
            'essentiality_minimal': essentiality_minimal,
            'essentiality_lb': essentiality_lb
        })

# Insert missing genes as new rows
insert_count = 0
for gene_info in genes_to_insert:
    cursor.execute("""
        INSERT INTO genome_features (genome_id, contig_id, feature_id, length, essentiality_minimal, essentiality_lb)
        VALUES ('user_Acinetobacter_baylyi_ADP1_RAST', 'unknown', ?, 0, ?, ?)
    """, (gene_info['gene_id'], gene_info['essentiality_minimal'], gene_info['essentiality_lb']))
    insert_count += 1

conn.commit()

print(f"\nUpdated {update_count} existing rows in genome_features")
print(f"Inserted {insert_count} new rows for missing genes")

# Show sample of updated data
sample_df = pd.read_sql("""
    SELECT feature_id, gene_names, essentiality_minimal, essentiality_lb 
    FROM genome_features 
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
    AND essentiality_minimal IS NOT NULL 
    LIMIT 10
""", conn)
print("\nSample updated genome_features:")
print(sample_df)

# Show sample of newly inserted genes
if insert_count > 0:
    new_genes_df = pd.read_sql("""
        SELECT feature_id, contig_id, essentiality_minimal, essentiality_lb 
        FROM genome_features 
        WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
        AND contig_id = 'unknown'
        LIMIT 5
    """, conn)
    print("\nSample newly inserted genes:")
    print(new_genes_df)

conn.close()

session.cache.save('essentiality_added_to_genome_features', {
    'total_genes': len(gene_essentiality),
    'updated_rows': update_count,
    'inserted_rows': insert_count
})

## Load Mutant Growth Rate Data

Load mutant growth rate measurements from the "Mutant growth phenotypes" sheet of MutantGrowthRatesData.xls. This data provides experimental growth rates for gene knockout mutants on various carbon sources.

The gene IDs in this file use the old ACIAD format (e.g., ACIAD0003) which will be translated to the new ACIAD_RS format using gene_translation.json.

In [ ]:
%run util.py



# Load gene translation mapping (old ACIAD -> new ACIAD_RS format)
gene_translation = session.cache.load('gene_translation')
print(f"Loaded gene translation with {len(gene_translation)} mappings")

# Load mutant growth rates from the correct sheet
mutant_file = "data/MutantGrowthRatesData.xls"

try:
    mutant_df = pd.read_excel(mutant_file, sheet_name='Mutant growth phenotypes', engine='xlrd')
except ImportError:
    print("xlrd library not available. Please install: pip install xlrd")
    mutant_df = None
except Exception as e:
    print(f"Error loading mutant growth rates: {e}")
    mutant_df = None

if mutant_df is not None:
    print(f"\nMutant growth phenotypes loaded: {mutant_df.shape}")
    print(f"Columns: {list(mutant_df.columns)}")
    print(f"\nCarbon sources: {[c for c in mutant_df.columns if c != 'Locus tag']}")
    
    # Translate gene IDs from old format to new format
    def translate_gene_id(old_id):
        """Translate old ACIAD format to new ACIAD_RS format."""
        if pd.isna(old_id):
            return None
        old_id_str = str(old_id).strip()
        # Try direct lookup
        if old_id_str in gene_translation:
            return gene_translation[old_id_str]
        # Try without leading zeros (e.g., ACIAD3 -> ACIAD0003)
        return None
    
    mutant_df['feature_id'] = mutant_df['Locus tag'].apply(translate_gene_id)
    
    # Count translations
    translated_count = mutant_df['feature_id'].notna().sum()
    print(f"\nTranslated {translated_count}/{len(mutant_df)} gene IDs to new format")
    
    # Show sample
    print("\nSample data with translated IDs:")
    print(mutant_df[['Locus tag', 'feature_id', 'acetate', 'glucose']].head(10))
    
    # Save info for next cell
    session.cache.save('mutant_growth_loaded', {
        'shape': list(mutant_df.shape),
        'columns': list(mutant_df.columns),
        'carbon_sources': [c for c in mutant_df.columns if c not in ['Locus tag', 'feature_id']],
        'translated_count': int(translated_count),
        'total_count': len(mutant_df)
    })
else:
    session.cache.save('mutant_growth_loaded', {'error': 'Could not load file'})

## Add Mutant Growth Rates to genome_features Table

Add mutant growth rate data directly to the genome_features table. For genes not found in the table, add them as new rows.

In [ ]:
%run util.py
import sqlite3



# Load mutant growth rate info
mutant_info = session.cache.load('mutant_growth_loaded')

if 'error' in mutant_info:
    print(f"Cannot process mutant growth rates: {mutant_info['error']}")
    session.cache.save('mutant_growth_added_to_genome_features', {'status': 'skipped', 'reason': mutant_info['error']})
else:
    # Reload the data and gene translation
    gene_translation = session.cache.load('gene_translation')
    
    mutant_file = "data/MutantGrowthRatesData.xls"
    mutant_df = pd.read_excel(mutant_file, sheet_name='Mutant growth phenotypes', engine='xlrd')
    
    # Translate gene IDs
    def translate_gene_id(old_id):
        if pd.isna(old_id):
            return None
        old_id_str = str(old_id).strip()
        return gene_translation.get(old_id_str)
    
    mutant_df['feature_id'] = mutant_df['Locus tag'].apply(translate_gene_id)
    
    print(f"Mutant growth phenotypes: {len(mutant_df)} entries")
    carbon_sources = [c for c in mutant_df.columns if c not in ['Locus tag', 'feature_id']]
    print(f"Carbon sources: {carbon_sources}")
    
    # Connect to database
    db_path = "data/berdl_tables.db"
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Delete mutant_growth_rates reference table if it exists
    conn.execute("DROP TABLE IF EXISTS mutant_growth_rates")
    print("Removed mutant_growth_rates reference table (if existed)")
    
    # Get existing feature_ids
    cursor.execute("SELECT feature_id FROM genome_features WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'")
    existing_features = set(row[0] for row in cursor.fetchall())
    print(f"Existing features in genome_features: {len(existing_features)}")
    
    # Add mutant growth rate columns to genome_features
    cursor.execute("PRAGMA table_info(genome_features)")
    existing_cols = [col[1] for col in cursor.fetchall()]
    
    # Create column names for each carbon source
    growth_cols = {}
    for carbon_source in carbon_sources:
        safe_col = f"mutant_growth_{carbon_source.replace(' ', '_').replace('-', '_').lower()}"
        growth_cols[carbon_source] = safe_col
        if safe_col not in existing_cols:
            conn.execute(f"ALTER TABLE genome_features ADD COLUMN {safe_col} REAL")
            print(f"Added {safe_col} column to genome_features")
    
    # Also add old_locus_tag column to store the original ID
    if 'old_locus_tag' not in existing_cols:
        conn.execute("ALTER TABLE genome_features ADD COLUMN old_locus_tag VARCHAR(50)")
        print("Added old_locus_tag column to genome_features")
    
    # Map mutant data to genome_features
    update_count = 0
    insert_count = 0
    not_found_count = 0
    
    for _, row in mutant_df.iterrows():
        old_locus_tag = str(row['Locus tag']).strip() if pd.notna(row['Locus tag']) else None
        feature_id = row['feature_id']
        
        if not old_locus_tag:
            continue
        
        # Build update values for all carbon sources
        update_values = {'old_locus_tag': old_locus_tag}
        for carbon_source, col_name in growth_cols.items():
            if pd.notna(row[carbon_source]):
                update_values[col_name] = float(row[carbon_source])
        
        if feature_id and feature_id in existing_features:
            # Update existing row
            set_clause = ', '.join([f"{k} = ?" for k in update_values.keys()])
            values = list(update_values.values()) + [feature_id]
            cursor.execute(f"""
                UPDATE genome_features 
                SET {set_clause}
                WHERE feature_id = ? AND genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
            """, values)
            if cursor.rowcount > 0:
                update_count += 1
        else:
            # Insert new row for genes not in genome_features
            # Use old locus tag as feature_id if translation failed
            new_feature_id = feature_id if feature_id else f"mutant_{old_locus_tag}"
            
            cols = ['genome_id', 'contig_id', 'feature_id', 'length'] + list(update_values.keys())
            vals = ['user_Acinetobacter_baylyi_ADP1_RAST', 'mutant_data', new_feature_id, 0] + list(update_values.values())
            
            placeholders = ', '.join(['?' for _ in vals])
            col_names = ', '.join(cols)
            
            try:
                cursor.execute(f"INSERT INTO genome_features ({col_names}) VALUES ({placeholders})", vals)
                insert_count += 1
            except sqlite3.IntegrityError:
                not_found_count += 1
    
    conn.commit()
    
    print(f"\nUpdated {update_count} existing rows in genome_features with mutant growth data")
    print(f"Inserted {insert_count} new rows for genes not in genome_features")
    if not_found_count > 0:
        print(f"Skipped {not_found_count} rows due to constraint violations")
    
    # Show sample
    sample_cols = ', '.join(['feature_id', 'old_locus_tag'] + list(growth_cols.values())[:3])
    sample_df = pd.read_sql(f"""
        SELECT {sample_cols}
        FROM genome_features 
        WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
        AND old_locus_tag IS NOT NULL 
        LIMIT 10
    """, conn)
    print("\nSample genome_features with mutant growth data:")
    print(sample_df)
    
    conn.close()
    
    session.cache.save('mutant_growth_added_to_genome_features', {
        'total_mutant_genes': len(mutant_df),
        'updated_rows': update_count,
        'inserted_rows': insert_count,
        'carbon_sources': carbon_sources,
        'columns_added': list(growth_cols.values()) + ['old_locus_tag']
    })

## Load Gene Phenotype Data from Database

Load fitness scores and essentiality fractions from the gene_phenotypes table for all phenotypes.

In [ ]:
%run util.py
import sqlite3


# Connect to database
db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)

# Get list of phenotypes
phenotypes_df = pd.read_sql("""
    SELECT DISTINCT phenotype_id, phenotype_name, COUNT(*) as gene_count
    FROM gene_phenotypes
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
    GROUP BY phenotype_id, phenotype_name
    ORDER BY phenotype_name
""", conn)

print(f"Total phenotypes: {len(phenotypes_df)}")
print("\nSample phenotypes:")
print(phenotypes_df.head(20))

# Load all gene phenotype data for ADP1
gene_phenotypes_df = pd.read_sql("""
    SELECT gene_id, phenotype_id, phenotype_name, fitness_avg, essentiality_fraction
    FROM gene_phenotypes
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
""", conn)

print(f"\nTotal gene-phenotype associations: {len(gene_phenotypes_df)}")
print(f"Unique genes: {gene_phenotypes_df['gene_id'].nunique()}")
print(f"Unique phenotypes: {gene_phenotypes_df['phenotype_id'].nunique()}")

# Show fitness statistics
print("\nFitness statistics (excluding nulls):")
fitness_stats = gene_phenotypes_df['fitness_avg'].describe()
print(fitness_stats)

conn.close()

session.cache.save('gene_phenotypes_loaded', {
    'total_associations': len(gene_phenotypes_df),
    'unique_genes': int(gene_phenotypes_df['gene_id'].nunique()),
    'unique_phenotypes': int(gene_phenotypes_df['phenotype_id'].nunique()),
    'phenotype_list': phenotypes_df['phenotype_id'].tolist()
})

## Correlate Proteomics with Fitness Scores

For each strain's proteomics data, correlate protein expression levels with fitness scores for each phenotype.

We need to:
1. Map ACIAD gene IDs to feature IDs (ACIAD_RSxxxxx format)
2. Calculate correlation between expression and fitness for each phenotype
3. Identify which phenotypes show strongest correlation

In [ ]:
%run util.py
import sqlite3


# Connect to database
db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)

# Load proteomics data from genome_features
proteomics_df = pd.read_sql("""
    SELECT feature_id, proteomics_gene_id,
           proteomics_avg_ADP1, proteomics_avg_ACN2586, proteomics_avg_ACN2821,
           proteomics_avg_ACN3425, proteomics_avg_ACN3427, proteomics_avg_ACN3429, proteomics_avg_ACN3430
    FROM genome_features 
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
    AND proteomics_avg_ADP1 IS NOT NULL
""", conn)
print(f"Proteomics data from genome_features: {len(proteomics_df)} genes")

# Get all feature_ids for mapping verification
all_features = pd.read_sql("""
    SELECT feature_id FROM genome_features 
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
""", conn)
print(f"Total features in genome_features: {len(all_features)}")

conn.close()

session.cache.save('gene_mapping_created', {
    'total_proteomics_genes': len(proteomics_df),
    'mapped_genes': len(proteomics_df)  # All are now mapped since they're in genome_features
})

## Calculate Correlations: Proteomics vs Fitness

Calculate Pearson and Spearman correlations between strain-averaged proteomics expression and fitness scores for each phenotype.

In [ ]:
%run util.py
import sqlite3
from scipy.stats import pearsonr, spearmanr


from scipy.stats import pearsonr, spearmanr

# Connect to database
db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)

# Load proteomics data from genome_features
proteomics_df = pd.read_sql("""
    SELECT feature_id, proteomics_gene_id,
           proteomics_avg_ADP1, proteomics_avg_ACN2586, proteomics_avg_ACN2821,
           proteomics_avg_ACN3425, proteomics_avg_ACN3427, proteomics_avg_ACN3429, proteomics_avg_ACN3430
    FROM genome_features 
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
    AND proteomics_avg_ADP1 IS NOT NULL
""", conn)

# Load gene phenotypes
gene_phenotypes_df = pd.read_sql("""
    SELECT gene_id, phenotype_id, phenotype_name, fitness_avg
    FROM gene_phenotypes
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
    AND fitness_avg IS NOT NULL
""", conn)

# Get list of phenotypes and strains
phenotypes = gene_phenotypes_df['phenotype_id'].unique()
strains = ['ADP1', 'ACN2586', 'ACN2821', 'ACN3425', 'ACN3427', 'ACN3429', 'ACN3430']

print(f"Calculating correlations for {len(phenotypes)} phenotypes and {len(strains)} strains...")

# Calculate correlations
correlations = []

for strain in strains:
    avg_col = f'proteomics_avg_{strain}'
    if avg_col not in proteomics_df.columns:
        continue
    
    # Get proteomics data for this strain (already has feature_id)
    strain_proteomics = proteomics_df[['feature_id', avg_col]].copy()
    strain_proteomics = strain_proteomics.dropna()
    
    for phenotype_id in phenotypes:
        # Get fitness data for this phenotype
        pheno_fitness = gene_phenotypes_df[gene_phenotypes_df['phenotype_id'] == phenotype_id][['gene_id', 'fitness_avg', 'phenotype_name']].copy()
        
        # Merge proteomics and fitness data
        merged = strain_proteomics.merge(
            pheno_fitness, 
            left_on='feature_id', 
            right_on='gene_id',
            how='inner'
        )
        
        if len(merged) >= 10:  # Need enough points for correlation
            try:
                pearson_r, pearson_p = pearsonr(merged[avg_col], merged['fitness_avg'])
                spearman_r, spearman_p = spearmanr(merged[avg_col], merged['fitness_avg'])
                
                correlations.append({
                    'strain': strain,
                    'phenotype_id': phenotype_id,
                    'phenotype_name': merged['phenotype_name'].iloc[0],
                    'n_genes': len(merged),
                    'pearson_r': pearson_r,
                    'pearson_p': pearson_p,
                    'spearman_r': spearman_r,
                    'spearman_p': spearman_p
                })
            except Exception as e:
                pass

correlations_df = pd.DataFrame(correlations)
print(f"\nCalculated {len(correlations_df)} correlations")

# Find best correlations per strain
print("\nTop 5 correlations per strain (by absolute Spearman r):")
for strain in strains:
    strain_corr = correlations_df[correlations_df['strain'] == strain].copy()
    if len(strain_corr) > 0:
        strain_corr['abs_spearman'] = strain_corr['spearman_r'].abs()
        top5 = strain_corr.nlargest(5, 'abs_spearman')[['phenotype_name', 'spearman_r', 'spearman_p', 'n_genes']]
        print(f"\n{strain}:")
        print(top5.to_string())

conn.close()

# Save results
session.cache.save('proteomics_fitness_correlations', {
    'total_correlations': len(correlations_df),
    'strains': strains,
    'top_correlations': correlations_df.nlargest(20, correlations_df['spearman_r'].abs())[['strain', 'phenotype_name', 'spearman_r']].to_dict('records') if len(correlations_df) > 0 else []
})

## Correlate Essentiality with Phenotype Predictions

Calculate correlation between experimental essentiality classifications and the essentiality_fraction from gene_phenotypes. Since essentiality is the same across phenotypes, we only need to compute this once per gene.

In [ ]:
%run util.py
import sqlite3
from scipy.stats import pearsonr, spearmanr, chi2_contingency


from scipy.stats import pearsonr, spearmanr, chi2_contingency

# Connect to database
db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)

# Load experimental essentiality
essentiality_df = pd.read_sql("SELECT * FROM gene_essentiality", conn)
print(f"Experimental essentiality: {len(essentiality_df)} genes")

# Load predicted essentiality fractions (unique per gene across phenotypes)
# Take one phenotype since essentiality_fraction should be same across phenotypes
predicted_df = pd.read_sql("""
    SELECT DISTINCT gene_id, essentiality_fraction
    FROM gene_phenotypes
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
    AND essentiality_fraction IS NOT NULL
""", conn)
print(f"Predicted essentiality: {len(predicted_df)} genes")

# Merge experimental and predicted
merged = essentiality_df.merge(predicted_df, on='gene_id', how='inner')
print(f"\nMerged genes: {len(merged)}")

# Convert experimental essentiality to numeric for correlation
# essential=1, uncertain=0.5, dispensable=0
essentiality_map = {'essential': 1.0, 'uncertain': 0.5, 'dispensable': 0.0}

merged['minimal_numeric'] = merged['essentiality_minimal'].map(essentiality_map)
merged['lb_numeric'] = merged['essentiality_lb'].map(essentiality_map)

# Calculate correlations for minimal media
minimal_valid = merged.dropna(subset=['minimal_numeric', 'essentiality_fraction'])
if len(minimal_valid) >= 10:
    pearson_r, pearson_p = pearsonr(minimal_valid['minimal_numeric'], minimal_valid['essentiality_fraction'])
    spearman_r, spearman_p = spearmanr(minimal_valid['minimal_numeric'], minimal_valid['essentiality_fraction'])
    print(f"\nMinimal media essentiality correlation (n={len(minimal_valid)}):")
    print(f"  Pearson r: {pearson_r:.4f} (p={pearson_p:.2e})")
    print(f"  Spearman r: {spearman_r:.4f} (p={spearman_p:.2e})")

# Calculate correlations for LB media
lb_valid = merged.dropna(subset=['lb_numeric', 'essentiality_fraction'])
if len(lb_valid) >= 10:
    pearson_r_lb, pearson_p_lb = pearsonr(lb_valid['lb_numeric'], lb_valid['essentiality_fraction'])
    spearman_r_lb, spearman_p_lb = spearmanr(lb_valid['lb_numeric'], lb_valid['essentiality_fraction'])
    print(f"\nLB media essentiality correlation (n={len(lb_valid)}):")
    print(f"  Pearson r: {pearson_r_lb:.4f} (p={pearson_p_lb:.2e})")
    print(f"  Spearman r: {spearman_r_lb:.4f} (p={spearman_p_lb:.2e})")

# Cross-tabulation analysis
print("\nCross-tabulation: Experimental vs Predicted (Minimal Media)")
# Bin predicted essentiality fraction
minimal_valid['pred_category'] = pd.cut(
    minimal_valid['essentiality_fraction'], 
    bins=[-0.01, 0.2, 0.8, 1.01], 
    labels=['predicted_dispensable', 'predicted_uncertain', 'predicted_essential']
)
crosstab = pd.crosstab(minimal_valid['essentiality_minimal'], minimal_valid['pred_category'])
print(crosstab)

conn.close()

session.cache.save('essentiality_correlation', {
    'minimal_n': len(minimal_valid) if len(minimal_valid) >= 10 else 0,
    'minimal_spearman_r': float(spearman_r) if len(minimal_valid) >= 10 else None,
    'lb_n': len(lb_valid) if len(lb_valid) >= 10 else 0,
    'lb_spearman_r': float(spearman_r_lb) if len(lb_valid) >= 10 else None
})

## Correlate Mutant Growth Rates with Fitness Scores

Calculate correlation between experimental mutant growth rates and model-predicted fitness scores for each phenotype.

In [ ]:
%run util.py
import sqlite3
from scipy.stats import pearsonr, spearmanr


from scipy.stats import pearsonr, spearmanr

# Connect to database
db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Check what mutant growth columns exist in genome_features
cursor.execute("PRAGMA table_info(genome_features)")
all_cols = [col[1] for col in cursor.fetchall()]
mutant_cols = [c for c in all_cols if c.startswith('mutant_growth_')]

if not mutant_cols:
    print("No mutant growth rate columns found in genome_features. Skipping correlation.")
    session.cache.save('mutant_fitness_correlations', {'status': 'skipped', 'reason': 'no mutant growth columns'})
else:
    print(f"Found mutant growth columns: {mutant_cols}")
    
    # Load mutant growth data from genome_features
    cols_to_select = ['feature_id', 'old_locus_tag'] + mutant_cols
    mutant_df = pd.read_sql(f"""
        SELECT {', '.join(cols_to_select)}
        FROM genome_features 
        WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
        AND old_locus_tag IS NOT NULL
    """, conn)
    print(f"\nMutant growth data: {len(mutant_df)} genes")
    
    # Load gene phenotypes with fitness
    gene_phenotypes_df = pd.read_sql("""
        SELECT gene_id, phenotype_id, phenotype_name, fitness_avg
        FROM gene_phenotypes
        WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
        AND fitness_avg IS NOT NULL
    """, conn)
    
    phenotypes = gene_phenotypes_df['phenotype_id'].unique()
    print(f"Calculating correlations for {len(phenotypes)} phenotypes and {len(mutant_cols)} carbon sources...")
    
    correlations = []
    
    for mutant_col in mutant_cols:
        # Extract carbon source name from column name (e.g., mutant_growth_acetate -> acetate)
        carbon_source = mutant_col.replace('mutant_growth_', '')
        
        # Get mutant growth data for this carbon source
        growth_data = mutant_df[['feature_id', mutant_col]].copy()
        growth_data = growth_data.dropna()
        
        for phenotype_id in phenotypes:
            pheno_fitness = gene_phenotypes_df[gene_phenotypes_df['phenotype_id'] == phenotype_id]
            
            # Merge on feature_id/gene_id
            merged = growth_data.merge(
                pheno_fitness,
                left_on='feature_id',
                right_on='gene_id',
                how='inner'
            )
            
            if len(merged) >= 10:
                try:
                    pearson_r, pearson_p = pearsonr(merged[mutant_col], merged['fitness_avg'])
                    spearman_r, spearman_p = spearmanr(merged[mutant_col], merged['fitness_avg'])
                    
                    correlations.append({
                        'carbon_source': carbon_source,
                        'phenotype_id': phenotype_id,
                        'phenotype_name': merged['phenotype_name'].iloc[0],
                        'n_genes': len(merged),
                        'pearson_r': pearson_r,
                        'pearson_p': pearson_p,
                        'spearman_r': spearman_r,
                        'spearman_p': spearman_p
                    })
                except:
                    pass
    
    correlations_df = pd.DataFrame(correlations)
    print(f"\nCalculated {len(correlations_df)} correlations")
    
    if len(correlations_df) > 0:
        # Find best correlations per carbon source
        print("\nTop 5 correlations per carbon source (by absolute Spearman r):")
        for carbon_source in mutant_cols:
            cs_name = carbon_source.replace('mutant_growth_', '')
            cs_corr = correlations_df[correlations_df['carbon_source'] == cs_name].copy()
            if len(cs_corr) > 0:
                cs_corr['abs_spearman'] = cs_corr['spearman_r'].abs()
                top5 = cs_corr.nlargest(5, 'abs_spearman')[['phenotype_name', 'spearman_r', 'spearman_p', 'n_genes']]
                print(f"\n{cs_name}:")
                print(top5.to_string())
        
        session.cache.save('mutant_fitness_correlations', {
            'total_correlations': len(correlations_df),
            'carbon_sources': [c.replace('mutant_growth_', '') for c in mutant_cols],
            'top_correlations': correlations_df.nlargest(20, correlations_df['spearman_r'].abs())[['carbon_source', 'phenotype_name', 'spearman_r', 'n_genes']].to_dict('records')
        })
    else:
        session.cache.save('mutant_fitness_correlations', {'status': 'no_correlations', 'reason': 'no sufficient overlap'})

conn.close()

## Summary: Best Phenotype Correlations

Summarize which phenotypes show the best correlations with each experimental data source.

In [ ]:
%run util.py


print("="*80)
print("SUMMARY: Best Phenotype Correlations by Data Source")
print("="*80)

# Load saved correlation results
proteomics_corr = session.cache.load('proteomics_fitness_correlations')
essentiality_corr = session.cache.load('essentiality_correlation')
mutant_corr = session.cache.load('mutant_fitness_correlations')

print("\n1. PROTEOMICS vs FITNESS CORRELATIONS")
print("-"*40)
if proteomics_corr and 'top_correlations' in proteomics_corr:
    print(f"Total correlations calculated: {proteomics_corr.get('total_correlations', 'N/A')}")
    print("\nTop correlations:")
    for i, corr in enumerate(proteomics_corr['top_correlations'][:10], 1):
        print(f"  {i}. {corr['strain']} - {corr['phenotype_name']}: r={corr['spearman_r']:.4f}")
else:
    print("No proteomics correlations available")

print("\n2. ESSENTIALITY CORRELATIONS")
print("-"*40)
if essentiality_corr:
    if essentiality_corr.get('minimal_n', 0) > 0:
        print(f"Minimal media: n={essentiality_corr['minimal_n']}, Spearman r={essentiality_corr['minimal_spearman_r']:.4f}")
    if essentiality_corr.get('lb_n', 0) > 0:
        print(f"LB media: n={essentiality_corr['lb_n']}, Spearman r={essentiality_corr['lb_spearman_r']:.4f}")
else:
    print("No essentiality correlations available")

print("\n3. MUTANT GROWTH RATE vs FITNESS CORRELATIONS")
print("-"*40)
if mutant_corr and 'top_correlations' in mutant_corr:
    print(f"Total correlations calculated: {mutant_corr.get('total_correlations', 'N/A')}")
    print(f"Carbon sources tested: {mutant_corr.get('carbon_sources', [])}")
    print("\nTop correlations:")
    for i, corr in enumerate(mutant_corr['top_correlations'][:10], 1):
        print(f"  {i}. {corr['carbon_source']} - {corr['phenotype_name']}: r={corr['spearman_r']:.4f} (n={corr['n_genes']})")
elif mutant_corr:
    print(f"Status: {mutant_corr.get('status', 'unknown')}, Reason: {mutant_corr.get('reason', 'N/A')}")
else:
    print("No mutant growth rate correlations available")

print("\n" + "="*80)

# Save final summary
session.cache.save('correlation_summary', {
    'proteomics_correlations': proteomics_corr,
    'essentiality_correlations': essentiality_corr,
    'mutant_growth_correlations': mutant_corr
})

## Export Correlation Results to Excel

Save all correlation results to an Excel file for further analysis.

In [ ]:
%run util.py
import sqlite3
from scipy.stats import pearsonr, spearmanr


from scipy.stats import pearsonr, spearmanr

# Rebuild full correlations DataFrames
db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)

# Load proteomics data from genome_features
proteomics_df = pd.read_sql("""
    SELECT feature_id, proteomics_gene_id,
           proteomics_avg_ADP1, proteomics_avg_ACN2586, proteomics_avg_ACN2821,
           proteomics_avg_ACN3425, proteomics_avg_ACN3427, proteomics_avg_ACN3429, proteomics_avg_ACN3430
    FROM genome_features 
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
    AND proteomics_avg_ADP1 IS NOT NULL
""", conn)

# Load phenotypes
gene_phenotypes_df = pd.read_sql("""
    SELECT gene_id, phenotype_id, phenotype_name, fitness_avg
    FROM gene_phenotypes
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
    AND fitness_avg IS NOT NULL
""", conn)

phenotypes = gene_phenotypes_df['phenotype_id'].unique()
strains = ['ADP1', 'ACN2586', 'ACN2821', 'ACN3425', 'ACN3427', 'ACN3429', 'ACN3430']

# Calculate all correlations
correlations = []
for strain in strains:
    avg_col = f'proteomics_avg_{strain}'
    if avg_col not in proteomics_df.columns:
        continue
    
    strain_proteomics = proteomics_df[['feature_id', avg_col]].copy()
    strain_proteomics = strain_proteomics.dropna()
    
    for phenotype_id in phenotypes:
        pheno_fitness = gene_phenotypes_df[gene_phenotypes_df['phenotype_id'] == phenotype_id][['gene_id', 'fitness_avg', 'phenotype_name']].copy()
        merged = strain_proteomics.merge(pheno_fitness, left_on='feature_id', right_on='gene_id', how='inner')
        
        if len(merged) >= 10:
            try:
                pearson_r, pearson_p = pearsonr(merged[avg_col], merged['fitness_avg'])
                spearman_r, spearman_p = spearmanr(merged[avg_col], merged['fitness_avg'])
                correlations.append({
                    'strain': strain,
                    'phenotype_id': phenotype_id,
                    'phenotype_name': merged['phenotype_name'].iloc[0],
                    'n_genes': len(merged),
                    'pearson_r': pearson_r,
                    'pearson_p': pearson_p,
                    'spearman_r': spearman_r,
                    'spearman_p': spearman_p
                })
            except:
                pass

correlations_df = pd.DataFrame(correlations)

# Export to Excel
output_file = "nboutput/ADP1BERDLAnalysis_correlations.xlsx"
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    if len(correlations_df) > 0:
        correlations_df.to_excel(writer, sheet_name='Proteomics_Fitness', index=False)
    
    # Add summary sheet
    summary_rows = []
    for strain in strains:
        strain_corr = correlations_df[correlations_df['strain'] == strain]
        if len(strain_corr) > 0:
            best = strain_corr.loc[strain_corr['spearman_r'].abs().idxmax()]
            summary_rows.append({
                'strain': strain,
                'best_phenotype': best['phenotype_name'],
                'spearman_r': best['spearman_r'],
                'n_genes': best['n_genes']
            })
    
    if summary_rows:
        summary_df = pd.DataFrame(summary_rows)
        summary_df.to_excel(writer, sheet_name='Summary', index=False)

print(f"Correlation results exported to: {output_file}")
print(f"Total rows in Proteomics_Fitness sheet: {len(correlations_df)}")

conn.close()

session.cache.save('excel_export', {'file': output_file, 'rows': len(correlations_df)})

## Gapfill Gene Candidate Analysis

Identify gene candidates for gapfilled reactions in the model by:
1. Building a list of gapfilled reactions (no gene associations, excluding exchange/biomass reactions)
2. Gathering EC and KO annotations from genome_features for ADP1 genes
3. Using AnnotationOntology to map annotations to ModelSEED reactions
4. Cross-referencing with the gapfilled reaction list
5. Harvesting additional annotations from pangenome cluster members
6. Generating a report of resolved gapfills from genome-only vs pangenome-augmented annotations

In [ ]:
%run util.py
import sqlite3
from modelseedpy.core.annotationontology import AnnotationOntology, AnnotationOntologyEvent


# ModelSEEDpy is available via venv
from modelseedpy.core.annotationontology import AnnotationOntology, AnnotationOntologyEvent

# Connect to database
db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)

# Step 1: Identify gapfilled reactions (no genes, not exchange/biomass)
all_rxns_df = pd.read_sql("""
    SELECT reaction_id, genes, gapfilling_status
    FROM genome_reactions
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
""", conn)

# Gapfilled reactions: empty genes column, excluding EX_, SK_, DM_, bio1
gapfilled_df = all_rxns_df[
    (all_rxns_df['genes'].isna() | (all_rxns_df['genes'] == '')) &
    (~all_rxns_df['reaction_id'].str.startswith('EX_')) &
    (~all_rxns_df['reaction_id'].str.startswith('SK_')) &
    (~all_rxns_df['reaction_id'].str.startswith('DM_')) &
    (all_rxns_df['reaction_id'] != 'bio1')
]

gapfilled_rxns = set(gapfilled_df['reaction_id'].tolist())
print(f"Total reactions in model: {len(all_rxns_df)}")
print(f"Gapfilled reactions (excluding exchange/biomass): {len(gapfilled_rxns)}")
print(f"  Minimal media gapfills: {len(gapfilled_df[gapfilled_df['gapfilling_status'] == 'minimal'])}")
print(f"  Rich media gapfills: {len(gapfilled_df[gapfilled_df['gapfilling_status'] == 'rich'])}")

# Step 2: Load EC and KO to reaction mappings from cb_annotation_ontology_api
data_dir = '/Users/chenry/Dropbox/Projects/cb_annotation_ontology_api/data'

# Build EC -> reaction mapping
ec_to_rxns = {}
with open(f'{data_dir}/EC_translation.tsv') as f:
    lines = f.read().strip().split('\n')
    for line in lines[1:]:  # Skip header
        items = line.split('\t')
        if len(items) >= 2:
            rxn_id = items[0]
            ec_num = items[1]
            ec_key = f'EC:{ec_num}'
            if ec_key not in ec_to_rxns:
                ec_to_rxns[ec_key] = []
            ec_to_rxns[ec_key].append(f'MSRXN:{rxn_id}')

print(f"\nEC to reaction mappings: {len(ec_to_rxns)} EC numbers -> reactions")

# Build KO -> reaction mapping
ko_to_rxns = {}
with open(f'{data_dir}/kegg_95_0_ko_seed.tsv') as f:
    lines = f.read().strip().split('\n')
    for line in lines[1:]:  # Skip header
        items = line.split('\t')
        if len(items) >= 2:
            ko_id = items[0]
            rxn_ids = items[1].split(';')
            ko_key = f'KO:{ko_id}'
            ko_to_rxns[ko_key] = [f'MSRXN:{r.strip()}' for r in rxn_ids if r.strip()]

print(f"KO to reaction mappings: {len(ko_to_rxns)} KO terms -> reactions")

# Step 3: Gather annotations from genome_features
annotations_df = pd.read_sql("""
    SELECT feature_id, ec, ko, pangenome_cluster_id
    FROM genome_features
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
""", conn)

# Build EC event data
ec_terms = {}  # feature_id -> list of term dicts
for _, row in annotations_df.iterrows():
    if pd.notna(row['ec']) and str(row['ec']).strip():
        feature_id = row['feature_id']
        ec_values = str(row['ec']).split(';')
        for ec_val in ec_values:
            ec_val = ec_val.strip()
            if not ec_val:
                continue
            # Normalize: add EC: prefix if not present
            ec_key = ec_val if ec_val.startswith('EC:') else f'EC:{ec_val}'
            if ec_key in ec_to_rxns:
                if feature_id not in ec_terms:
                    ec_terms[feature_id] = []
                ec_terms[feature_id].append({
                    'term': ec_key,
                    'modelseed_ids': ec_to_rxns[ec_key]
                })

# Build KO event data
ko_terms = {}  # feature_id -> list of term dicts
for _, row in annotations_df.iterrows():
    if pd.notna(row['ko']) and str(row['ko']).strip():
        feature_id = row['feature_id']
        ko_values = str(row['ko']).split(';')
        for ko_val in ko_values:
            ko_val = ko_val.strip()
            if not ko_val:
                continue
            # Remove KEGG: prefix if present, then add KO: prefix
            if ko_val.startswith('KEGG:'):
                ko_val = ko_val[5:]
            ko_key = ko_val if ko_val.startswith('KO:') else f'KO:{ko_val}'
            if ko_key in ko_to_rxns:
                if feature_id not in ko_terms:
                    ko_terms[feature_id] = []
                ko_terms[feature_id].append({
                    'term': ko_key,
                    'modelseed_ids': ko_to_rxns[ko_key]
                })

print(f"\nGenome annotations with EC->reaction mappings: {len(ec_terms)} genes")
print(f"Genome annotations with KO->reaction mappings: {len(ko_terms)} genes")

# Step 4: Build AnnotationOntology and get reaction-gene hash
anno = AnnotationOntology('user_Acinetobacter_baylyi_ADP1_RAST', data_dir)

# Create EC event
if ec_terms:
    ec_event_data = {
        'event_id': 'genome_ec',
        'ontology_id': 'EC',
        'method': 'genome_ec_annotations',
        'ontology_terms': ec_terms
    }
    AnnotationOntologyEvent.from_data(ec_event_data, anno)

# Create KO event
if ko_terms:
    ko_event_data = {
        'event_id': 'genome_ko',
        'ontology_id': 'KO',
        'method': 'genome_ko_annotations',
        'ontology_terms': ko_terms
    }
    AnnotationOntologyEvent.from_data(ko_event_data, anno)

# Get reaction-gene hash
genome_rxn_gene_hash = anno.get_reaction_gene_hash(merge_all=True)

print(f"\nReactions found from genome annotations: {len(genome_rxn_gene_hash)}")

# Cross-reference with gapfilled reactions
genome_resolved_gapfills = {}
genome_new_reactions = {}
for rxn_id, gene_data in genome_rxn_gene_hash.items():
    if rxn_id in gapfilled_rxns:
        genome_resolved_gapfills[rxn_id] = gene_data
    # Check if this is a reaction not currently in the model at all
    if rxn_id not in set(all_rxns_df['reaction_id']):
        genome_new_reactions[rxn_id] = gene_data

print(f"Gapfilled reactions resolved by genome annotations: {len(genome_resolved_gapfills)}")
print(f"New reactions (not in model) from genome annotations: {len(genome_new_reactions)}")

if genome_resolved_gapfills:
    print("\nResolved gapfills from genome annotations:")
    for rxn_id, genes in sorted(genome_resolved_gapfills.items()):
        gene_list = ', '.join(sorted(genes.keys()))
        print(f"  {rxn_id}: {gene_list}")

conn.close()

session.cache.save('ADP1BERDLAnalysis/genome_gapfill_analysis', {
    'gapfilled_rxns': list(gapfilled_rxns),
    'genome_total_annotation_rxns': len(genome_rxn_gene_hash),
    'genome_resolved_gapfills': {k: list(v.keys()) for k, v in genome_resolved_gapfills.items()},
    'genome_new_reactions': len(genome_new_reactions)
})

## Pangenome-Augmented Gapfill Gene Candidates

For each ADP1 gene with a pangenome cluster assignment, harvest EC and KO annotations from all other genes in the same cluster. These additional annotations may provide evidence for gapfilled reactions that the genome annotations alone cannot resolve.

In [ ]:
%run util.py
import sqlite3
from modelseedpy.core.annotationontology import AnnotationOntology, AnnotationOntologyEvent


# ModelSEEDpy is available via venv
from modelseedpy.core.annotationontology import AnnotationOntology, AnnotationOntologyEvent

# Load saved genome analysis results
genome_results = session.cache.load('ADP1BERDLAnalysis/genome_gapfill_analysis')
gapfilled_rxns = set(genome_results['gapfilled_rxns'])

# Load EC and KO to reaction mappings
data_dir = '/Users/chenry/Dropbox/Projects/cb_annotation_ontology_api/data'

ec_to_rxns = {}
with open(f'{data_dir}/EC_translation.tsv') as f:
    lines = f.read().strip().split('\n')
    for line in lines[1:]:
        items = line.split('\t')
        if len(items) >= 2:
            ec_key = f'EC:{items[1]}'
            if ec_key not in ec_to_rxns:
                ec_to_rxns[ec_key] = []
            ec_to_rxns[ec_key].append(f'MSRXN:{items[0]}')

ko_to_rxns = {}
with open(f'{data_dir}/kegg_95_0_ko_seed.tsv') as f:
    lines = f.read().strip().split('\n')
    for line in lines[1:]:
        items = line.split('\t')
        if len(items) >= 2:
            ko_key = f'KO:{items[0]}'
            ko_to_rxns[ko_key] = [f'MSRXN:{r.strip()}' for r in items[1].split(';') if r.strip()]

# Connect to database
db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)

# Get all reaction_ids in model
all_rxns_df = pd.read_sql("""
    SELECT reaction_id FROM genome_reactions
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
""", conn)
all_model_rxns = set(all_rxns_df['reaction_id'])

# Get ADP1 genes with pangenome cluster assignments
adp1_clusters_df = pd.read_sql("""
    SELECT feature_id, pangenome_cluster_id
    FROM genome_features
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
    AND pangenome_cluster_id IS NOT NULL AND pangenome_cluster_id != ''
""", conn)

print(f"ADP1 genes with pangenome clusters: {len(adp1_clusters_df)}")

# Build cluster_id -> ADP1 feature_id mapping
cluster_to_adp1 = {}
for _, row in adp1_clusters_df.iterrows():
    cid = row['pangenome_cluster_id']
    if cid not in cluster_to_adp1:
        cluster_to_adp1[cid] = []
    cluster_to_adp1[cid].append(row['feature_id'])

print(f"Unique pangenome clusters for ADP1: {len(cluster_to_adp1)}")

# Get pangenome annotations for all clusters that contain ADP1 genes
cluster_ids = list(cluster_to_adp1.keys())
# Query in batches to avoid SQL limits
batch_size = 500
pan_annotations = []
for i in range(0, len(cluster_ids), batch_size):
    batch = cluster_ids[i:i+batch_size]
    placeholders = ','.join(['?' for _ in batch])
    batch_df = pd.read_sql(f"""
        SELECT genome_id, feature_id, cluster_id, ec, ko
        FROM pan_genome_features
        WHERE cluster_id IN ({placeholders})
        AND genome_id != 'user_Acinetobacter_baylyi_ADP1_RAST'
    """, conn, params=batch)
    pan_annotations.append(batch_df)

pan_df = pd.concat(pan_annotations, ignore_index=True) if pan_annotations else pd.DataFrame()
print(f"Pangenome genes in ADP1 clusters (non-ADP1): {len(pan_df)}")
print(f"Unique genomes contributing: {pan_df['genome_id'].nunique()}")

# Build pangenome-augmented annotations: map pangenome EC/KO -> ADP1 feature_ids
# For each pangenome gene, harvest its annotations and attribute them to the ADP1 gene(s) in the same cluster
pan_ec_terms = {}  # ADP1 feature_id -> list of term dicts
pan_ko_terms = {}

for _, row in pan_df.iterrows():
    cluster_id = row['cluster_id']
    adp1_features = cluster_to_adp1.get(cluster_id, [])
    
    # Parse EC annotations
    if pd.notna(row['ec']) and str(row['ec']).strip():
        ec_values = str(row['ec']).split(';')
        for ec_val in ec_values:
            ec_val = ec_val.strip()
            if not ec_val:
                continue
            if ec_val.startswith('EC:'):
                ec_key = ec_val
            else:
                ec_key = f'EC:{ec_val}'
            if ec_key in ec_to_rxns:
                for adp1_fid in adp1_features:
                    if adp1_fid not in pan_ec_terms:
                        pan_ec_terms[adp1_fid] = []
                    pan_ec_terms[adp1_fid].append({
                        'term': ec_key,
                        'modelseed_ids': ec_to_rxns[ec_key]
                    })
    
    # Parse KO annotations
    if pd.notna(row['ko']) and str(row['ko']).strip():
        ko_values = str(row['ko']).split(';')
        for ko_val in ko_values:
            ko_val = ko_val.strip()
            if not ko_val:
                continue
            if ko_val.startswith('KEGG:'):
                ko_val = ko_val[5:]
            ko_key = ko_val if ko_val.startswith('KO:') else f'KO:{ko_val}'
            if ko_key in ko_to_rxns:
                for adp1_fid in adp1_features:
                    if adp1_fid not in pan_ko_terms:
                        pan_ko_terms[adp1_fid] = []
                    pan_ko_terms[adp1_fid].append({
                        'term': ko_key,
                        'modelseed_ids': ko_to_rxns[ko_key]
                    })

print(f"\nPangenome EC annotations mapped to ADP1 genes: {len(pan_ec_terms)} genes")
print(f"Pangenome KO annotations mapped to ADP1 genes: {len(pan_ko_terms)} genes")

# Also include original genome annotations (combine genome + pangenome)
# Reload genome annotations
genome_annotations_df = pd.read_sql("""
    SELECT feature_id, ec, ko
    FROM genome_features
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
""", conn)

# Merge genome EC annotations into pangenome set
combined_ec_terms = dict(pan_ec_terms)  # Start with pangenome
for _, row in genome_annotations_df.iterrows():
    if pd.notna(row['ec']) and str(row['ec']).strip():
        feature_id = row['feature_id']
        ec_values = str(row['ec']).split(';')
        for ec_val in ec_values:
            ec_val = ec_val.strip()
            if not ec_val:
                continue
            ec_key = ec_val if ec_val.startswith('EC:') else f'EC:{ec_val}'
            if ec_key in ec_to_rxns:
                if feature_id not in combined_ec_terms:
                    combined_ec_terms[feature_id] = []
                combined_ec_terms[feature_id].append({
                    'term': ec_key,
                    'modelseed_ids': ec_to_rxns[ec_key]
                })

combined_ko_terms = dict(pan_ko_terms)
for _, row in genome_annotations_df.iterrows():
    if pd.notna(row['ko']) and str(row['ko']).strip():
        feature_id = row['feature_id']
        ko_values = str(row['ko']).split(';')
        for ko_val in ko_values:
            ko_val = ko_val.strip()
            if not ko_val:
                continue
            if ko_val.startswith('KEGG:'):
                ko_val = ko_val[5:]
            ko_key = ko_val if ko_val.startswith('KO:') else f'KO:{ko_val}'
            if ko_key in ko_to_rxns:
                if feature_id not in combined_ko_terms:
                    combined_ko_terms[feature_id] = []
                combined_ko_terms[feature_id].append({
                    'term': ko_key,
                    'modelseed_ids': ko_to_rxns[ko_key]
                })

print(f"\nCombined (genome+pangenome) EC annotations: {len(combined_ec_terms)} genes")
print(f"Combined (genome+pangenome) KO annotations: {len(combined_ko_terms)} genes")

# Build combined AnnotationOntology
combined_anno = AnnotationOntology('user_Acinetobacter_baylyi_ADP1_RAST', data_dir)

if combined_ec_terms:
    ec_event_data = {
        'event_id': 'combined_ec',
        'ontology_id': 'EC',
        'method': 'genome_plus_pangenome_ec',
        'ontology_terms': combined_ec_terms
    }
    AnnotationOntologyEvent.from_data(ec_event_data, combined_anno)

if combined_ko_terms:
    ko_event_data = {
        'event_id': 'combined_ko',
        'ontology_id': 'KO',
        'method': 'genome_plus_pangenome_ko',
        'ontology_terms': combined_ko_terms
    }
    AnnotationOntologyEvent.from_data(ko_event_data, combined_anno)

# Get reaction-gene hash from combined annotations
combined_rxn_gene_hash = combined_anno.get_reaction_gene_hash(merge_all=True)

print(f"\nReactions from combined annotations: {len(combined_rxn_gene_hash)}")

# Cross-reference with gapfilled reactions
combined_resolved_gapfills = {}
combined_new_reactions = {}
for rxn_id, gene_data in combined_rxn_gene_hash.items():
    if rxn_id in gapfilled_rxns:
        combined_resolved_gapfills[rxn_id] = gene_data
    if rxn_id not in all_model_rxns:
        combined_new_reactions[rxn_id] = gene_data

print(f"Gapfilled reactions resolved by combined annotations: {len(combined_resolved_gapfills)}")
print(f"New reactions (not in model) from combined annotations: {len(combined_new_reactions)}")

# Identify gapfills resolved ONLY by pangenome (not by genome alone)
genome_resolved = set(genome_results['genome_resolved_gapfills'].keys())
pan_only_resolved = {k: v for k, v in combined_resolved_gapfills.items() if k not in genome_resolved}
print(f"\nGapfills resolved ONLY by pangenome annotations (not genome): {len(pan_only_resolved)}")

if pan_only_resolved:
    print("\nPangenome-only resolved gapfills:")
    for rxn_id, genes in sorted(pan_only_resolved.items()):
        gene_list = ', '.join(sorted(genes.keys()))
        print(f"  {rxn_id}: {gene_list}")

conn.close()

session.cache.save('ADP1BERDLAnalysis/pangenome_gapfill_analysis', {
    'pangenome_genes_harvested': len(pan_df),
    'unique_genomes': int(pan_df['genome_id'].nunique()) if len(pan_df) > 0 else 0,
    'combined_total_annotation_rxns': len(combined_rxn_gene_hash),
    'combined_resolved_gapfills': {k: list(v.keys()) for k, v in combined_resolved_gapfills.items()},
    'combined_new_reactions': len(combined_new_reactions),
    'pangenome_only_resolved': {k: list(v.keys()) for k, v in pan_only_resolved.items()}
})

## Gapfill Gene Candidate Report

Summary report comparing genome-only vs pangenome-augmented gapfill resolution, with a detailed table of all resolved gapfills and their candidate genes.

In [ ]:
%run util.py


# Load both analysis results
genome_results = session.cache.load('ADP1BERDLAnalysis/genome_gapfill_analysis')
pan_results = session.cache.load('ADP1BERDLAnalysis/pangenome_gapfill_analysis')

print("=" * 80)
print("GAPFILL GENE CANDIDATE ANALYSIS REPORT")
print("=" * 80)

gapfilled_rxns = genome_results['gapfilled_rxns']
genome_resolved = genome_results['genome_resolved_gapfills']
genome_total_rxns = genome_results['genome_total_annotation_rxns']
genome_new_rxns = genome_results['genome_new_reactions']

combined_resolved = pan_results['combined_resolved_gapfills']
combined_total_rxns = pan_results['combined_total_annotation_rxns']
combined_new_rxns = pan_results['combined_new_reactions']
pan_only_resolved = pan_results['pangenome_only_resolved']
pan_genes_harvested = pan_results['pangenome_genes_harvested']
unique_genomes = pan_results['unique_genomes']

print(f"\n--- Model Overview ---")
print(f"Total gapfilled reactions (excl. exchange/biomass): {len(gapfilled_rxns)}")

print(f"\n--- Genome Annotations Only ---")
print(f"Total reactions mapped from genome EC/KO annotations: {genome_total_rxns}")
print(f"Gapfilled reactions resolved: {len(genome_resolved)} / {len(gapfilled_rxns)} ({100*len(genome_resolved)/len(gapfilled_rxns):.1f}%)")
print(f"New reactions found (not in model): {genome_new_rxns}")

print(f"\n--- Pangenome-Augmented Annotations ---")
print(f"Pangenome genes harvested from clusters: {pan_genes_harvested}")
print(f"Unique genomes contributing annotations: {unique_genomes}")
print(f"Total reactions mapped from combined annotations: {combined_total_rxns}")
print(f"Gapfilled reactions resolved: {len(combined_resolved)} / {len(gapfilled_rxns)} ({100*len(combined_resolved)/len(gapfilled_rxns):.1f}%)")
print(f"New reactions found (not in model): {combined_new_rxns}")
print(f"Additional gapfills resolved by pangenome only: {len(pan_only_resolved)}")

# Build detailed table of all resolved gapfills
print(f"\n--- Detailed Resolved Gapfills ---")
print(f"{'Reaction':<15} {'Source':<12} {'Candidate Genes'}")
print("-" * 80)

all_resolved = set(list(genome_resolved.keys()) + list(combined_resolved.keys()))
for rxn_id in sorted(all_resolved):
    if rxn_id in genome_resolved:
        source = "Genome"
        genes = genome_resolved[rxn_id]
    else:
        source = "Pangenome"
        genes = combined_resolved[rxn_id]
    gene_str = ', '.join(sorted(genes))
    print(f"{rxn_id:<15} {source:<12} {gene_str}")

# Create a DataFrame for export
resolved_rows = []
for rxn_id in sorted(all_resolved):
    in_genome = rxn_id in genome_resolved
    in_pangenome = rxn_id in combined_resolved
    
    genome_genes = genome_resolved.get(rxn_id, [])
    combined_genes = combined_resolved.get(rxn_id, [])
    
    resolved_rows.append({
        'reaction_id': rxn_id,
        'source': 'genome' if in_genome else 'pangenome_only',
        'genome_candidate_genes': '; '.join(sorted(genome_genes)) if genome_genes else '',
        'combined_candidate_genes': '; '.join(sorted(combined_genes)) if combined_genes else '',
        'pangenome_added_genes': '; '.join(sorted(set(combined_genes) - set(genome_genes))) if combined_genes else ''
    })

resolved_df = pd.DataFrame(resolved_rows)

# Export to CSV in nboutput
output_dir = 'nboutput/ADP1BERDLAnalysis'
os.makedirs(output_dir, exist_ok=True)
output_file = f'{output_dir}/gapfill_gene_candidates.csv'
resolved_df.to_csv(output_file, index=False)
print(f"\nDetailed results exported to: {output_file}")
print(f"Total resolved gapfills in export: {len(resolved_df)}")

# Display the DataFrame
print("\nResolved gapfills table:")
display(resolved_df)

session.cache.save('ADP1BERDLAnalysis/gapfill_report', {
    'total_gapfilled': len(gapfilled_rxns),
    'genome_resolved_count': len(genome_resolved),
    'combined_resolved_count': len(combined_resolved),
    'pangenome_only_count': len(pan_only_resolved),
    'genome_total_annotation_rxns': genome_total_rxns,
    'combined_total_annotation_rxns': combined_total_rxns,
    'export_file': output_file
})

## Functional Category Charts: Gapfilled vs Resolved Reactions

Classify gapfilled reactions and resolved gapfills by their ModelSEED subsystem functional categories. This uses the ModelSEED_Subsystems.tsv file to map reaction IDs to metabolic pathway classes.

In [ ]:
%run util.py
import matplotlib.pyplot as plt
import matplotlib


import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 6)

# Load gapfill analysis results
genome_results = session.cache.load('ADP1BERDLAnalysis/genome_gapfill_analysis')
pan_results = session.cache.load('ADP1BERDLAnalysis/pangenome_gapfill_analysis')
gapfilled_rxns = set(genome_results['gapfilled_rxns'])
genome_resolved = set(genome_results['genome_resolved_gapfills'].keys())
combined_resolved = set(pan_results['combined_resolved_gapfills'].keys())

# Load ModelSEED subsystems for functional categories
subsys_df = pd.read_csv(
    '/Users/chenry/Dropbox/Projects/ModelSEEDDatabase/Biochemistry/Pathways/ModelSEED_Subsystems.tsv',
    sep='\t'
)

# Build reaction -> class mapping (take first class if multiple)
rxn_to_class = {}
for _, row in subsys_df.iterrows():
    rxn = row['Reaction']
    cls = row['Class']
    if rxn not in rxn_to_class:
        rxn_to_class[rxn] = cls

# Load curated ReactionClasses overrides from cache
curated_classes = {}
overrides_applied = 0
reaction_classes = session.cache.load('ADP1BERDLAnalysis/ReactionClasses', default=None)
if reaction_classes is not None:
    curated_classes = reaction_classes
    # Apply curated overrides (only non-empty values)
    for rxn_id, cls in curated_classes.items():
        if cls and cls.strip():
            rxn_to_class[rxn_id] = cls.strip()
            overrides_applied += 1
    print(f"Loaded {len(curated_classes)} curated reaction classes, {overrides_applied} non-empty overrides applied")
else:
    print("No curated ReactionClasses found in cache")

# Classify all gapfilled reactions
gapfill_classes = []
for rxn in gapfilled_rxns:
    cls = rxn_to_class.get(rxn, 'Unclassified')
    if cls == '-':
        cls = 'Unclassified'
    gapfill_classes.append(cls)

gapfill_class_counts = pd.Series(gapfill_classes).value_counts()

# Classify resolved gapfills (combined = genome + pangenome)
resolved_classes = []
for rxn in combined_resolved:
    cls = rxn_to_class.get(rxn, 'Unclassified')
    if cls == '-':
        cls = 'Unclassified'
    resolved_classes.append(cls)

resolved_class_counts = pd.Series(resolved_classes).value_counts()

# Create side-by-side bar charts
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Chart 1: All gapfilled reactions by functional category
colors1 = plt.cm.Set3(range(len(gapfill_class_counts)))
bars1 = axes[0].barh(range(len(gapfill_class_counts)), gapfill_class_counts.values, color=colors1)
axes[0].set_yticks(range(len(gapfill_class_counts)))
axes[0].set_yticklabels(gapfill_class_counts.index, fontsize=9)
axes[0].set_xlabel('Number of Reactions')
axes[0].set_title(f'All Gapfilled Reactions by Functional Category\n(n={len(gapfilled_rxns)})')
axes[0].invert_yaxis()
for i, v in enumerate(gapfill_class_counts.values):
    axes[0].text(v + 0.3, i, str(v), va='center', fontsize=9)

# Chart 2: Resolved gapfills by functional category
if len(resolved_class_counts) > 0:
    colors2 = plt.cm.Set2(range(len(resolved_class_counts)))
    bars2 = axes[1].barh(range(len(resolved_class_counts)), resolved_class_counts.values, color=colors2)
    axes[1].set_yticks(range(len(resolved_class_counts)))
    axes[1].set_yticklabels(resolved_class_counts.index, fontsize=9)
    axes[1].set_xlabel('Number of Reactions')
    axes[1].set_title(f'Resolved Gapfills by Functional Category\n(n={len(combined_resolved)})')
    axes[1].invert_yaxis()
    for i, v in enumerate(resolved_class_counts.values):
        axes[1].text(v + 0.1, i, str(v), va='center', fontsize=9)
else:
    axes[1].text(0.5, 0.5, 'No resolved gapfills', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Resolved Gapfills by Functional Category')

plt.tight_layout()
os.makedirs('nboutput/ADP1BERDLAnalysis', exist_ok=True)
plt.savefig('nboutput/ADP1BERDLAnalysis/gapfill_functional_categories.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary
print(f"\nAll gapfilled reactions ({len(gapfilled_rxns)}):")
for cls, count in gapfill_class_counts.items():
    print(f"  {cls}: {count}")

print(f"\nResolved gapfills ({len(combined_resolved)}):")
for cls, count in resolved_class_counts.items():
    print(f"  {cls}: {count}")

# Show resolution rate by category
print(f"\nResolution rate by category:")
for cls in gapfill_class_counts.index:
    total = gapfill_class_counts[cls]
    resolved = resolved_class_counts.get(cls, 0)
    pct = 100 * resolved / total if total > 0 else 0
    print(f"  {cls}: {resolved}/{total} ({pct:.0f}%)")

# List any still-unclassified reactions
still_unclassified = [r for r in gapfilled_rxns if rxn_to_class.get(r, 'Unclassified') in ('Unclassified', '-')]
if still_unclassified:
    print(f"\nStill unclassified ({len(still_unclassified)}): {', '.join(sorted(still_unclassified))}")

session.cache.save('ADP1BERDLAnalysis/gapfill_functional_categories', {
    'gapfill_categories': gapfill_class_counts.to_dict(),
    'resolved_categories': resolved_class_counts.to_dict(),
    'curated_overrides_applied': overrides_applied
})

## Gene Candidate Fitness and Essentiality Analysis

For the resolved gapfills, analyze whether gene candidates:
1. Have significant negative fitness values for the pyruvate (cpd00020) phenotype in gene_phenotypes
2. Are classified as essential in minimal media from the essentiality data

In [ ]:
%run util.py
import matplotlib.pyplot as plt
import sqlite3


import matplotlib.pyplot as plt

# Load resolved gapfill results
genome_results = session.cache.load('ADP1BERDLAnalysis/genome_gapfill_analysis')
pan_results = session.cache.load('ADP1BERDLAnalysis/pangenome_gapfill_analysis')

# Collect all unique gene candidates from combined resolved gapfills
combined_resolved = pan_results['combined_resolved_gapfills']
all_candidate_genes = set()
for rxn_id, genes in combined_resolved.items():
    all_candidate_genes.update(genes)

print(f"Total unique gene candidates across all resolved gapfills: {len(all_candidate_genes)}")

# Connect to database
db_path = "data/berdl_tables.db"
conn = sqlite3.connect(db_path)

# --- 1. Pyruvate phenotype fitness analysis ---
# Get fitness data for pyruvate phenotype (cpd00020)
pyruvate_fitness = pd.read_sql("""
    SELECT gene_id, fitness_avg, essentiality_fraction
    FROM gene_phenotypes
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
    AND phenotype_id = 'cpd00020'
""", conn)

print(f"\nGenes with pyruvate fitness data: {len(pyruvate_fitness)}")

# Match gene candidates to pyruvate fitness
candidate_fitness = pyruvate_fitness[pyruvate_fitness['gene_id'].isin(all_candidate_genes)].copy()
print(f"Gene candidates with pyruvate fitness data: {len(candidate_fitness)}")

# Define "significant negative fitness" threshold
# fitness_avg < -0.1 is a common threshold for growth defect
fitness_threshold = -0.1
neg_fitness = candidate_fitness[candidate_fitness['fitness_avg'] < fitness_threshold]
print(f"Gene candidates with significant negative fitness (< {fitness_threshold}): {len(neg_fitness)}")

if len(neg_fitness) > 0:
    print("\nGene candidates with negative pyruvate fitness:")
    for _, row in neg_fitness.sort_values('fitness_avg').iterrows():
        # Find which reactions this gene resolves
        rxns_for_gene = [rxn for rxn, genes in combined_resolved.items() if row['gene_id'] in genes]
        print(f"  {row['gene_id']}: fitness={row['fitness_avg']:.4f}, essentiality_frac={row['essentiality_fraction']:.4f}, resolves: {', '.join(rxns_for_gene)}")

# --- 2. Essentiality analysis ---
# Get essentiality data from genome_features
essentiality_df = pd.read_sql("""
    SELECT feature_id, essentiality_minimal, essentiality_lb
    FROM genome_features
    WHERE genome_id = 'user_Acinetobacter_baylyi_ADP1_RAST'
    AND feature_id IN ({})
""".format(','.join([f"'{g}'" for g in all_candidate_genes])), conn)

print(f"\nGene candidates with essentiality data: {len(essentiality_df)}")

essential_minimal = essentiality_df[essentiality_df['essentiality_minimal'] == 'essential']
print(f"Gene candidates essential in minimal media: {len(essential_minimal)}")

if len(essential_minimal) > 0:
    print("\nEssential gene candidates (minimal media):")
    for _, row in essential_minimal.iterrows():
        rxns_for_gene = [rxn for rxn, genes in combined_resolved.items() if row['feature_id'] in genes]
        print(f"  {row['feature_id']}: essentiality_lb={row['essentiality_lb']}, resolves: {', '.join(rxns_for_gene)}")

# --- 3. Combined summary per resolved reaction ---
print("\n" + "="*80)
print("DETAILED ANALYSIS: Resolved Gapfills with Gene Candidate Phenotypes")
print("="*80)

# Build lookup dicts
pyruvate_fitness_dict = dict(zip(pyruvate_fitness['gene_id'], pyruvate_fitness['fitness_avg']))
essentiality_dict = dict(zip(essentiality_df['feature_id'], essentiality_df['essentiality_minimal']))

summary_rows = []
for rxn_id in sorted(combined_resolved.keys()):
    genes = combined_resolved[rxn_id]
    for gene in sorted(genes):
        fitness = pyruvate_fitness_dict.get(gene)
        ess = essentiality_dict.get(gene)
        has_neg_fitness = fitness is not None and fitness < fitness_threshold
        is_essential = ess == 'essential'
        source = 'genome' if rxn_id in genome_results['genome_resolved_gapfills'] else 'pangenome'
        
        summary_rows.append({
            'reaction_id': rxn_id,
            'gene_candidate': gene,
            'source': source,
            'pyruvate_fitness': fitness,
            'significant_neg_fitness': has_neg_fitness,
            'essentiality_minimal': ess,
            'is_essential_minimal': is_essential
        })

summary_df = pd.DataFrame(summary_rows)
print(f"\nTotal gene-reaction pairs: {len(summary_df)}")
print(f"Pairs with significant negative pyruvate fitness: {summary_df['significant_neg_fitness'].sum()}")
print(f"Pairs with essential gene in minimal media: {summary_df['is_essential_minimal'].sum()}")

# Show the full table
print("\nFull table:")
display(summary_df)

# --- 4. Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Chart A: Fitness distribution of gene candidates vs all genes
ax = axes[0]
all_fitness = pyruvate_fitness['fitness_avg'].dropna()
candidate_fit_vals = candidate_fitness['fitness_avg'].dropna()
ax.hist(all_fitness, bins=50, alpha=0.5, label=f'All genes (n={len(all_fitness)})', color='steelblue', density=True)
if len(candidate_fit_vals) > 0:
    ax.hist(candidate_fit_vals, bins=20, alpha=0.7, label=f'Gapfill candidates (n={len(candidate_fit_vals)})', color='coral', density=True)
ax.axvline(x=fitness_threshold, color='red', linestyle='--', alpha=0.7, label=f'Threshold ({fitness_threshold})')
ax.set_xlabel('Pyruvate Fitness (fitness_avg)')
ax.set_ylabel('Density')
ax.set_title('Pyruvate Fitness Distribution')
ax.legend(fontsize=8)

# Chart B: Pie chart of fitness categories for gene candidates
ax = axes[1]
if len(candidate_fitness) > 0:
    n_neg = len(neg_fitness)
    n_pos = len(candidate_fitness) - n_neg
    n_no_data = len(all_candidate_genes) - len(candidate_fitness)
    sizes = [n_neg, n_pos, n_no_data]
    labels = [f'Neg fitness\n(n={n_neg})', f'Non-neg fitness\n(n={n_pos})', f'No data\n(n={n_no_data})']
    colors = ['#e74c3c', '#2ecc71', '#95a5a6']
    # Only plot non-zero slices
    non_zero = [(s, l, c) for s, l, c in zip(sizes, labels, colors) if s > 0]
    if non_zero:
        ax.pie([x[0] for x in non_zero], labels=[x[1] for x in non_zero], 
               colors=[x[2] for x in non_zero], autopct='%1.0f%%', startangle=90)
ax.set_title(f'Gene Candidates: Pyruvate Fitness\n(n={len(all_candidate_genes)} genes)')

# Chart C: Pie chart of essentiality categories for gene candidates
ax = axes[2]
n_essential = len(essential_minimal)
n_uncertain = len(essentiality_df[essentiality_df['essentiality_minimal'] == 'uncertain'])
n_dispensable = len(essentiality_df[essentiality_df['essentiality_minimal'] == 'dispensable'])
n_no_ess_data = len(all_candidate_genes) - len(essentiality_df[essentiality_df['essentiality_minimal'].notna()])
sizes = [n_essential, n_uncertain, n_dispensable, n_no_ess_data]
labels = [f'Essential\n(n={n_essential})', f'Uncertain\n(n={n_uncertain})', 
          f'Dispensable\n(n={n_dispensable})', f'No data\n(n={n_no_ess_data})']
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#95a5a6']
non_zero = [(s, l, c) for s, l, c in zip(sizes, labels, colors) if s > 0]
if non_zero:
    ax.pie([x[0] for x in non_zero], labels=[x[1] for x in non_zero],
           colors=[x[2] for x in non_zero], autopct='%1.0f%%', startangle=90)
ax.set_title(f'Gene Candidates: Minimal Media Essentiality\n(n={len(all_candidate_genes)} genes)')

plt.tight_layout()
plt.savefig('nboutput/ADP1BERDLAnalysis/gapfill_candidate_phenotypes.png', dpi=150, bbox_inches='tight')
plt.show()

# Export detailed table
output_file = 'nboutput/ADP1BERDLAnalysis/gapfill_candidate_phenotypes.csv'
summary_df.to_csv(output_file, index=False)
print(f"\nDetailed results exported to: {output_file}")

conn.close()

session.cache.save('ADP1BERDLAnalysis/gapfill_candidate_phenotypes', {
    'total_candidate_genes': len(all_candidate_genes),
    'candidates_with_pyruvate_fitness': len(candidate_fitness),
    'candidates_neg_fitness': len(neg_fitness),
    'candidates_essential_minimal': len(essential_minimal),
    'fitness_threshold': fitness_threshold
})